In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
from PIL import Image as PILImage

# 1. Load Models
model_path = 'trainer.yml' 
# Ensure parameters match your training exactly (radius=1, neighbors=8 from our previous steps)
recognizer = cv2.face.LBPHFaceRecognizer_create(radius=1, neighbors=8, grid_x=8, grid_y=8)

try:
    recognizer.read(model_path)
    print(f"Model loaded successfully from {model_path}")
except Exception as e:
    print(f"Error loading model: {e}")

# --- CHANGE: Use 'frontalface_alt2' instead of 'default' ---
# 'alt2' is the best trade-off in OpenCV. It detects faces better than 'default'
# and makes fewer mistakes on hands/shadows.
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml')

# Fallback in case alt2 is missing on your system
if face_cascade.empty():
    print("Warning: 'alt2' cascade not found. Falling back to default.")
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# ID to Name Mapping
id_to_name = {
    1: "Besheer",
    2: "Ashraf",
    3: "Seif",
    4: "Sallam",
    5: "Roger",
    6: "Omar"
}

# 2. UI Elements
uploader = widgets.FileUpload(accept='image/*', multiple=False)
out_image = widgets.Output()
out_results = widgets.Output()

def process_image(change):
    out_image.clear_output()
    out_results.clear_output()
    
    if not uploader.value:
        return
        
    # Get uploaded file
    if isinstance(uploader.value, tuple):
        uploaded_file = uploader.value[0]
    else:
        uploaded_file = next(iter(uploader.value.values()))
        
    content = uploaded_file['content']
    if isinstance(content, memoryview):
        content = content.tobytes()
    
    # Convert to OpenCV format
    image = PILImage.open(io.BytesIO(content))
    img_np = np.array(image)
    
    # Convert to Gray for detection
    if len(img_np.shape) == 3:
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    else:
        img_bgr = img_np.copy()
        gray = img_np

    # 3. DETECTION PHASE (Optimized)
    # scaleFactor=1.05: Scans the image in smaller steps (slower, but catches more faces).
    # minNeighbors=4: Good balance for 'alt2'. 
    # Removed the "Eye Check" so we don't accidentally delete real faces.
    faces = face_cascade.detectMultiScale(
        gray, 
        scaleFactor=1.05, 
        minNeighbors=4,     
        minSize=(30, 30)
    )
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    raw_predictions = []
    
    print(f"Candidates Detected: {len(faces)}")

    for i, (x, y, w, h) in enumerate(faces):
        # --- RECOGNITION PHASE ---
        face_roi = gray[y:y+h, x:x+w]
        
        # Resize using Cubic Interpolation (Best for upscaling small faces)
        face_resized = cv2.resize(face_roi, (200, 200), interpolation=cv2.INTER_CUBIC)
        
        # Preprocessing (Must match training exactly)
        face_smooth = cv2.bilateralFilter(face_resized, 5, 75, 75)
        face_final = clahe.apply(face_smooth)
        
        # Predict
        predicted_id, conf = recognizer.predict(face_final)
        
        # Threshold Logic
        # If conf < 135, we trust it. 
        if conf <70:
            name = id_to_name.get(predicted_id, "Unknown")
        else:
            name = "Unknown"
        
        raw_predictions.append({
            'id': i,
            'bbox': (x, y, w, h),
            'predicted_name': name,
            'confidence': conf,
            'original_roi': img_np[y:y+h, x:x+w]
        })
        
    # --- DEDUPLICATION ---
    # Sort by best confidence (lowest number) first
    raw_predictions.sort(key=lambda x: x['confidence'])
    
    predictions = []
    seen_names = set()
    
    for pred in raw_predictions:
        name = pred['predicted_name']
        
        if name == "Unknown":
            predictions.append(pred)
        elif name in seen_names:
            pred['predicted_name'] = f"{name}?" # Mark duplicate as ambiguous
            predictions.append(pred)
        else:
            seen_names.add(name)
            predictions.append(pred)
            
    predictions.sort(key=lambda x: x['id']) # Restore spatial order

    # --- DISPLAY IMAGE ---
    with out_image:
        img_display = img_np.copy()
        
        if len(predictions) == 0:
            print("No faces detected.")
            
        for pred in predictions:
            x, y, w, h = pred['bbox']
            name = pred['predicted_name']
            conf = pred['confidence']
            
            if name == "Unknown":
                color = (255, 0, 0) # Red
            elif "?" in name:
                color = (0, 165, 255) # Orange
            else:
                color = (0, 255, 0) # Green
                
            cv2.rectangle(img_display, (x, y), (x+w, y+h), color, 2)
            cv2.putText(img_display, f"{name} ({int(conf)})", (x, y-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            
        display(PILImage.fromarray(img_display))
        print(f"Displaying {len(predictions)} faces.")

    # --- VERIFICATION UI ---
    with out_results:
        if not predictions:
            return
            
        rows = []
        user_inputs = []
        
        for pred in predictions:
            roi_bgr = cv2.cvtColor(pred['original_roi'], cv2.COLOR_RGB2BGR)
            thumb = widgets.Image(value=cv2.imencode('.png', roi_bgr)[1].tobytes(), 
                                  format='png', width=80)
            
            info = widgets.HTML(f"<b>Detected:</b> {pred['predicted_name']}<br><b>Conf:</b> {pred['confidence']:.2f}")
            
            clean_name = pred['predicted_name'].replace("?", "")
            options = list(id_to_name.values()) + ["Unknown", "Other"]
            default_val = clean_name if clean_name in options else "Unknown"
            
            dropdown = widgets.Dropdown(options=options, value=default_val, description='Actual:')
            user_inputs.append({'pred': pred, 'input': dropdown})
            
            rows.append(widgets.HBox([thumb, info, dropdown]))
            
        btn_calc = widgets.Button(description="Calculate Accuracy", button_style='success')
        lbl_acc = widgets.Label(value="")
        
        def on_calc_click(b):
            correct = 0
            for item in user_inputs:
                predicted_clean = item['pred']['predicted_name'].replace("?", "")
                if item['input'].value == predicted_clean:
                    correct += 1
            
            acc = (correct / len(user_inputs)) * 100
            lbl_acc.value = f"Accuracy: {acc:.2f}% ({correct}/{len(user_inputs)})"
            
        btn_calc.on_click(on_calc_click)
        display(widgets.VBox(rows + [widgets.HBox([btn_calc, lbl_acc])]))

uploader.observe(process_image, names='value')

display(widgets.VBox([
    widgets.HTML("<h2>Face Recognition Tester (Alt2 + Slower Scan)</h2>"),
    uploader, out_image, out_results
]))

Model loaded successfully from trainer.yml
